In [1]:
# Core libraries for data handling
import numpy as np
import pandas as pd

# Utilities for splitting and evaluating the model
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

# Preprocessing and modeling tools
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

# For saving the trained pipeline
import joblib

In [18]:
# Define the raw Cleveland dataset path (relative to project root)
data_path = "../Data_set/processed.cleveland.data"

# Read the whole raw file as text because the raw format is token-based and spans multiple lines
with open(data_path, "r", encoding="latin-1") as f:
    raw_text = f.read()

# Build records from comma-separated values line by line
records = []

for line in raw_text.splitlines():
    line = line.replace('\x00', '').strip()

    if line == "":
        continue

    tokens = line.split(",")

    row = []
    for token in tokens:
        token = token.strip()

        if token == "?":
            row.append(None)
        else:
            try:
                row.append(float(token))
            except ValueError:
                row.append(None)

    records.append(row)

print("Parsed records:", len(records))
# Basic check to ensure rows were extracted
print("Number of patient records parsed:", len(records))
print("Example row length:", len(records[0]) if len(records) > 0 else 0)

Parsed records: 303
Number of patient records parsed: 303
Example row length: 14


In [28]:
# Convert parsed records into a DataFrame
raw_df = pd.DataFrame(records)

# Attribute positions from heart-disease.names (1-based -> converted to 0-based)
# 3 age, 4 sex, 9 cp, 10 trestbps, 12 chol, 16 fbs, 19 restecg,
# 32 thalach, 38 exang, 40 oldpeak, 41 slope, 44 ca, 51 thal, 58 num(target)
selected_indices = [2, 3, 8, 9, 11, 15, 18, 31, 37, 39, 40, 43, 50, 57]

# Human-readable column names for the selected attributes
selected_columns = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
    "thalach", "exang", "oldpeak", "slope", "ca", "thal", "num"
]

# If records are already the 14 selected columns (as in processed.cleveland.data), use them directly.
# Otherwise, select by positional indices from a wider raw table.
if raw_df.shape[1] == len(selected_columns):
    df = raw_df.copy()
    df.columns = selected_columns
else:
    df = raw_df.iloc[:, selected_indices].copy()
    df.columns = selected_columns

# Replace raw missing indicator -9 with NaN for proper imputation later
df.replace(-9, np.nan, inplace=True)

# Convert multiclass diagnosis into binary classification:
# 0 means no heart disease, 1 means presence of heart disease
df["target"] = (df["num"] > 0).astype(int)

# Drop the original "num" column because "target" is now the prediction label
df.drop(columns=["num"], inplace=True)

# Show data snapshot and missing counts
print(df.head())
print("\nMissing values per column:\n", df.isnull().sum())
print("\nClass distribution:\n", df["target"].value_counts())

    age  sex   cp  trestbps   chol  fbs  restecg  thalach  exang  oldpeak  \
0  63.0  1.0  1.0     145.0  233.0  1.0      2.0    150.0    0.0      2.3   
1  67.0  1.0  4.0     160.0  286.0  0.0      2.0    108.0    1.0      1.5   
2  67.0  1.0  4.0     120.0  229.0  0.0      2.0    129.0    1.0      2.6   
3  37.0  1.0  3.0     130.0  250.0  0.0      0.0    187.0    0.0      3.5   
4  41.0  0.0  2.0     130.0  204.0  0.0      2.0    172.0    0.0      1.4   

   slope   ca  thal  target  
0    3.0  0.0   6.0       0  
1    2.0  3.0   3.0       1  
2    2.0  2.0   7.0       1  
3    3.0  0.0   3.0       0  
4    1.0  0.0   3.0       0  

Missing values per column:
 age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          4
thal        2
target      0
dtype: int64

Class distribution:
 target
0    164
1    139
Name: count, dtype: int64


In [ ]:
# Data cleaning: enforce numeric types and standardize invalid entries as missing values
feature_cols = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
    "thalach", "exang", "oldpeak", "slope", "ca", "thal"
]

df[feature_cols] = df[feature_cols].apply(pd.to_numeric, errors="coerce")

# Domain rules from Cleveland dataset coding
binary_cols = ["sex", "fbs", "exang"]
valid_binary = {0, 1}
for col in binary_cols:
    df.loc[~df[col].isin(valid_binary), col] = np.nan

valid_categories = {
    "cp": {1, 2, 3, 4},
    "restecg": {0, 1, 2},
    "slope": {1, 2, 3},
    "thal": {3, 6, 7},
}
for col, valid_set in valid_categories.items():
    df.loc[~df[col].isin(valid_set), col] = np.nan

# Physiologically impossible negative values are treated as missing
for col in ["age", "trestbps", "chol", "thalach", "oldpeak", "ca"]:
    df.loc[df[col] < 0, col] = np.nan

# Ensure target is strictly binary and remove duplicate records
df = df[df["target"].isin([0, 1])].copy()
rows_before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
rows_after = len(df)

print("Data cleaning completed.")
print("Rows before duplicate removal:", rows_before)
print("Rows after duplicate removal :", rows_after)
print("Duplicates removed           :", rows_before - rows_after)
print("\nMissing values after cleaning:\n", df.isnull().sum())

In [29]:
# Separate features (X) and label (y)
X = df.drop(columns=["target"])
y = df["target"]

# Split data while preserving class ratio using stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Build a pipeline:
# 1) Median imputation for missing values
# 2) Standardization for SVM
# 3) SVM classifier with probability enabled for ROC-AUC
svm_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf", C=1.0, gamma="scale", probability=True, class_weight="balanced", random_state=42))
])

# Train the pipeline on training data
svm_pipeline.fit(X_train, y_train)

print("Training completed.")

Training completed.


In [30]:
# Predict class labels on test set
y_pred = svm_pipeline.predict(X_test)

# Predict class probabilities for ROC-AUC (use probability of positive class)
y_prob = svm_pipeline.predict_proba(X_test)[:, 1]

# Compute key classification metrics
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

# Print metric summary
print("Baseline SVM Results")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-score : {f1:.4f}")
print(f"ROC-AUC  : {auc:.4f}")

# Print confusion matrix and full classification report
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Baseline SVM Results
Accuracy : 0.8689
Precision: 0.8125
Recall   : 0.9286
F1-score : 0.8667
ROC-AUC  : 0.9437

Confusion Matrix:
 [[27  6]
 [ 2 26]]

Classification Report:
               precision    recall  f1-score   support

           0       0.93      0.82      0.87        33
           1       0.81      0.93      0.87        28

    accuracy                           0.87        61
   macro avg       0.87      0.87      0.87        61
weighted avg       0.88      0.87      0.87        61



In [31]:
# Define parameter grid for SVM inside the pipeline
param_grid = {
    "svm__C": [0.1, 1, 10, 50],
    "svm__gamma": ["scale", 0.01, 0.1, 1],
    "svm__kernel": ["rbf", "linear"]
}

# Use stratified 5-fold CV for robust model selection
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Optimize for ROC-AUC because this is a medical classification problem
grid_search = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

# Fit grid search on training set only
grid_search.fit(X_train, y_train)

# Get the best model from CV
best_model = grid_search.best_estimator_

print("Best parameters:", grid_search.best_params_)
print("Best CV ROC-AUC:", grid_search.best_score_)

Fitting 5 folds for each of 32 candidates, totalling 160 fits
Best parameters: {'svm__C': 1, 'svm__gamma': 0.01, 'svm__kernel': 'rbf'}
Best CV ROC-AUC: 0.8981537785885612


In [32]:
# Predict with tuned model on test data
best_pred = best_model.predict(X_test)
best_prob = best_model.predict_proba(X_test)[:, 1]

# Compute tuned model metrics
best_acc = accuracy_score(y_test, best_pred)
best_prec = precision_score(y_test, best_pred)
best_rec = recall_score(y_test, best_pred)
best_f1 = f1_score(y_test, best_pred)
best_auc = roc_auc_score(y_test, best_prob)

# Print tuned performance
print("Tuned SVM Results")
print(f"Accuracy : {best_acc:.4f}")
print(f"Precision: {best_prec:.4f}")
print(f"Recall   : {best_rec:.4f}")
print(f"F1-score : {best_f1:.4f}")
print(f"ROC-AUC  : {best_auc:.4f}")

# Persist tuned model pipeline to disk for reuse
joblib.dump(best_model, "../models/svm_heart_disease_model.joblib")
print("Saved model to ../models/svm_heart_disease_model.joblib")

Tuned SVM Results
Accuracy : 0.8525
Precision: 0.8065
Recall   : 0.8929
F1-score : 0.8475
ROC-AUC  : 0.9372
Saved model to ../models/svm_heart_disease_model.joblib


In [40]:
# Create an example patient row using the same feature order as training columns
sample_patient = pd.DataFrame([{
    "age": 26,
    "sex": 1,
    "cp": 1,
    "trestbps": 130,
    "chol": 250,
    "fbs": 0,
    "restecg": 1,
    "thalach": 140,
    "exang": 1,
    "oldpeak": 2.0,
    "slope": 2,
    "ca": 1,
    "thal": 7
}])

# Predict class and probability using tuned model
sample_pred = best_model.predict(sample_patient)[0]
sample_prob = best_model.predict_proba(sample_patient)[0, 1]

# Display patient-level prediction
print("Predicted class (0=no disease, 1=disease):", int(sample_pred))
print("Predicted disease probability:", round(float(sample_prob), 4))
print("Interpretation: This patient has a {:.2%} chance of having heart disease.".format(sample_prob))
print("Model Accuracy on Test Set: {:.2%}".format(best_acc))

Predicted class (0=no disease, 1=disease): 1
Predicted disease probability: 0.769
Interpretation: This patient has a 76.90% chance of having heart disease.
Model Accuracy on Test Set: 85.25%


In [ ]:
#This is for the output of the SVM classification results and artifacts
# Save SVM artifacts and reports in outputs/svm_classification
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay

output_dir = Path("../outputs/svm_classification")
output_dir.mkdir(parents=True, exist_ok=True)

# Save a text summary report
summary_path = output_dir / "summary_report.txt"
with open(summary_path, "w", encoding="utf-8") as f:
    f.write("Heart Disease Prediction - SVM Classification Summary Report\n")
    f.write("=" * 68 + "\n")
    f.write(f"Best Parameters: {grid_search.best_params_}\n")
    f.write(f"Best CV ROC-AUC: {grid_search.best_score_:.4f}\n")
    f.write(f"Test Accuracy : {best_acc:.4f}\n")
    f.write(f"Test Precision: {best_prec:.4f}\n")
    f.write(f"Test Recall   : {best_rec:.4f}\n")
    f.write(f"Test F1-score : {best_f1:.4f}\n")
    f.write(f"Test ROC-AUC  : {best_auc:.4f}\n\n")
    f.write("Classification Report:\n")
    f.write(classification_report(y_test, best_pred))

# Save prediction-level outputs
predictions_df = X_test.copy()
predictions_df["actual"] = y_test.values
predictions_df["predicted"] = best_pred
predictions_df["probability"] = best_prob
predictions_df.to_csv(output_dir / "svm_predictions.csv", index=False)

# Save confusion matrix values as CSV
cm = confusion_matrix(y_test, best_pred)
pd.DataFrame(cm, index=["actual_0", "actual_1"], columns=["pred_0", "pred_1"]).to_csv(
    output_dir / "confusion_matrix.csv"
 )

# Save confusion matrix figure
disp_cm = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1])
disp_cm.plot(cmap="Blues", values_format="d")
plt.title("SVM Confusion Matrix")
plt.tight_layout()
plt.savefig(output_dir / "confusion_matrix.png", dpi=300)
plt.close()

# Save ROC curve figure
RocCurveDisplay.from_predictions(y_test, best_prob)
plt.title("SVM ROC Curve")
plt.tight_layout()
plt.savefig(output_dir / "roc_curve.png", dpi=300)
plt.close()

print("Saved SVM outputs to:", output_dir.resolve())
print("- summary_report.txt")
print("- svm_predictions.csv")
print("- confusion_matrix.csv")
print("- confusion_matrix.png")
print("- roc_curve.png")

Saved SVM outputs to: C:\Users\ASUS\Desktop\SLIIT\Year 4\Machine Learning - ML\Group Assignment\heart-disease-prediction\outputs\svm_classification
- summary_report.txt
- svm_predictions.csv
- confusion_matrix.csv
- confusion_matrix.png
- roc_curve.png
